In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over. This repository uses the flat layout
#: `<repo>/notebooks` -> `<repo>`, so the depth is 0.
LOCAL_ROOT_DEPTH = 0


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


# The metric pilot — Neighborhood Latent Consistency, r09 Eqs. (8)-(9)

Executed report of the pilot r09 § "Falsifiable claim and assumptions" (Metric
claim, zero-GT) leaves open: the **unit** (per-annotator Eq. (9) vs the first
prototype's panel-consensus variant), the **normalization** of C_a (N0 / S1 /
S2z / S2p / S3) and the **panel summary** (mean / median / q10 /
fraction-below). Every number here is PROVISIONAL and nothing is selected: the
selection closes in the operator's r10 deliberation, and this notebook only
measures.

**What is measured.** The prototype's own bench: 15 Inria patches, one
neighborhood, 10 annotators per panel; the contamination ladder replaces
(10-c) good (P1) annotators with c poor (P5) ones, c = 0..10 — 30 draws per
rung, the declared verdict floor, subsets drawn once (seed 42) and SHARED by
every arm. The P3 arm runs the identical ladder with the systematic-bias
profile: that is the declared scope measurement (consistency is not
correctness), reported apart.

**What counts as good.** A variant earns attention (never selection) by: a
dynamic range that opens over the ladder (the compression 0.7984->0.7865 is
the problem), monotone movement without beyond-noise rebounds, adjacent steps
that separate past 2 combined SEs (above all 9+1 vs 8+2), and early steps
already moving. A variant with no verdicts past the floor is reported as such.

**Ground truth enters only the validation column** (mIoU/Dice of the binary
consensus against the dataset GT, reported beside the scores); the decision
metrics are zero-GT and proven invariant to the validation column's values by
`test_pilot_supervised_column_validation_only`.

In [2]:
import json
import sys
import time

import torch

sys.path.insert(0, str(ROOT / "src"))

# The fingerprint of the code this report ran against — `report_digest` is the
# single implementation; `verify` recomputes and compares it.
from CrowdSegmenter_CT_GCE import report_digest
print(report_digest.stamp())

from CrowdSegmenter_CT_GCE.metric_pilot import MetricPilotConfig, VERDICT_FLOOR_DRAWS
from CrowdSegmenter_CT_GCE_Benchmark.metric_pilot_run import run_metric_pilot

CONFIG = MetricPilotConfig(
    data_dir=str(ROOT / "data"),
    annotators_dir=str(ROOT / "data" / "annotators"),
    prototype_csv=str(ROOT / "data" / "prototype_reference" / "contamination_ablation_results.csv"),
)
print(json.dumps({k: getattr(CONFIG, k) for k in CONFIG.__dataclass_fields__}, indent=1))

SOURCES-SHA256 c609eb535fa3deb66bc39f4305a1a14982a348013a3bb2c53d9cd0735f08de43
{
 "data_dir": "/home/lucas/papers/ct-gce-smith/implementations/CrowdSegmenter-CT-GCE/data",
 "annotators_dir": "/home/lucas/papers/ct-gce-smith/implementations/CrowdSegmenter-CT-GCE/data/annotators",
 "prototype_csv": "/home/lucas/papers/ct-gce-smith/implementations/CrowdSegmenter-CT-GCE/data/prototype_reference/contamination_ablation_results.csv",
 "base_seed": 42,
 "subset_seed": 42,
 "n_annotators": 10,
 "good_profile": "P1",
 "poor_profile": "P5",
 "bias_profile": "P3",
 "draws": 30,
 "quantile": 0.1,
 "threshold_k": 1.0,
 "n_random_masks": 32,
 "random_seed": 7,
 "disk_radius": 64
}


## Cost, measured before the grid

The grid commits only after one measured figure exists. The run is one
`run_metric_pilot` call; the record's own timings break it apart.

In [3]:
t0 = time.time()
record = run_metric_pilot(CONFIG)
total = time.time() - t0
print("measured wall time:", round(total, 1), "s")
for key, value in record["timings"].items():
    print(f"  {key}: {value:.1f} s")
print("annotator masks read from cache:", record["material"]["annotatorMasksReadFromCache"])
print("annotator masks generated (the P3 gap):", record["material"]["annotatorMasksGenerated"])

out_path = ROOT / "results" / "metric" / "metric_pilot.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(record, indent=1))
print("record written:", out_path)

measured wall time: 670.4 s
  featureExtraction: 5.2 s
  perAnnotatorScores: 4.8 s
  anchors: 5.6 s
  ladder_P5: 318.0 s
  ladder_P3: 321.4 s
  total: 670.3 s
annotator masks read from cache: 450
annotator masks generated (the P3 gap): 0
record written: /home/lucas/papers/ct-gce-smith/implementations/CrowdSegmenter-CT-GCE/results/metric/metric_pilot.json


## Port fidelity — the prototype's own 6-draw protocol

Before any variant is read, the port is checked against the table the
prototype published. The compositions below are the prototype's own (same
seed, same sampling sequence), so a difference is device/precision drift of
the port, never sampling. The tolerance is declared at 1e-3; the prototype's
CSV was produced on an unknown device and the port runs on CPU, so the
per-rung difference is reported either way.

In [4]:
fidelity = record["portFidelity"]
lines = ["| c | draws | port mean C (U2) | prototype C_k | abs diff |",
         "| ---: | ---: | ---: | ---: | ---: |"]
for row in fidelity["rows"]:
    ref = row["prototypeCk"]
    ref_s = "-" if ref is None else f"{ref:.6f}"
    diff = row["absDiff"]
    diff_s = "-" if diff is None else f"{diff:.2e}"
    lines.append(f"| {row['c']} | {row['nDraws']} | {row['portMean']:.6f} | {ref_s} | {diff_s} |")
print("\n".join(lines))
print()
print("max abs diff:", fidelity["maxAbsDiff"], "| declared tolerance:", fidelity["tolerance"])
assert fidelity["maxAbsDiff"] is not None, "the prototype CSV is missing: the fidelity pass cannot run"
if fidelity["maxAbsDiff"] > fidelity["tolerance"]:
    print()
    print("ABOVE THE DECLARED TOLERANCE: the port diverges from the prototype's "
          "table; read the per-rung column before trusting any variant below.")

| c | draws | port mean C (U2) | prototype C_k | abs diff |
| ---: | ---: | ---: | ---: | ---: |
| 0 | 1 | 0.799484 | 0.798424 | 1.06e-03 |
| 1 | 6 | 0.799299 | 0.797886 | 1.41e-03 |
| 2 | 6 | 0.798945 | 0.797482 | 1.46e-03 |
| 3 | 6 | 0.798684 | 0.797068 | 1.62e-03 |
| 4 | 6 | 0.798829 | 0.797311 | 1.52e-03 |
| 5 | 6 | 0.798213 | 0.795470 | 2.74e-03 |
| 6 | 6 | 0.797528 | 0.793769 | 3.76e-03 |
| 7 | 6 | 0.794937 | 0.790116 | 4.82e-03 |
| 8 | 6 | 0.794657 | 0.789529 | 5.13e-03 |
| 9 | 6 | 0.793644 | 0.788330 | 5.31e-03 |
| 10 | 1 | 0.792124 | 0.786525 | 5.60e-03 |

max abs diff: 0.005599142214561481 | declared tolerance: 0.001

ABOVE THE DECLARED TOLERANCE: the port diverges from the prototype's table; read the per-rung column before trusting any variant below.


## The unit's raw material — C_a per annotator, by profile

The per-annotator unit's own evidence, before any summary touches it: the 30
annotators' Eq. (9) values on the neighborhood. The unit hypothesis says the
good and poor pools separate HERE — that is the ladder opening, in its purest
form. The P3 pool sits beside them: its values are the scope measurement's
baseline (a consistent, biased annotator scores wherever consistency puts
it, however wrong it is).

In [5]:
import numpy as np

scores = record["perAnnotatorScores"]
pop_good = np.array([scores[f"P1|{a}"] for a in range(CONFIG.n_annotators)])
pop_poor = np.array([scores[f"P5|{a}"] for a in range(CONFIG.n_annotators)])
pop_bias = np.array([scores[f"P3|{a}"] for a in range(CONFIG.n_annotators)])

def profile_row(name, values):
    v = np.asarray(values, dtype=float)
    return f"| {name} | {v.mean():.4f} | {v.std(ddof=1):.4f} | {v.min():.4f} | {v.max():.4f} |"

lines = ["| profile | mean C_a | sd | min | max |", "| --- | ---: | ---: | ---: | ---: |",
         profile_row("P1 (expert)", pop_good), profile_row("P5 (poor)", pop_poor),
         profile_row("P3 (systematic bias)", pop_bias)]
print("\n".join(lines))
gap = pop_good.mean() - pop_poor.mean()
se = np.sqrt(pop_good.std(ddof=1) ** 2 / len(pop_good) + pop_poor.std(ddof=1) ** 2 / len(pop_poor))
print()
print(f"P1 - P5 gap: {gap:.4f} | combined SE of the pools: {se:.4f} | separation: {gap / se:.1f} SEs")
gap3 = pop_good.mean() - pop_bias.mean()
se3 = np.sqrt(pop_good.std(ddof=1) ** 2 / len(pop_good) + pop_bias.std(ddof=1) ** 2 / len(pop_bias))
print(f"P1 - P3 gap: {gap3:.4f} | separation: {gap3 / se3:.1f} SEs "
      "(the scope limit: how far consistent bias sits from the experts on this metric)")

| profile | mean C_a | sd | min | max |
| --- | ---: | ---: | ---: | ---: |
| P1 (expert) | 0.7999 | 0.0003 | 0.7992 | 0.8002 |
| P5 (poor) | 0.7708 | 0.0080 | 0.7525 | 0.7814 |
| P3 (systematic bias) | 0.7898 | 0.0023 | 0.7869 | 0.7926 |

P1 - P5 gap: 0.0291 | combined SE of the pools: 0.0025 | separation: 11.4 SEs
P1 - P3 gap: 0.0101 | separation: 14.0 SEs (the scope limit: how far consistent bias sits from the experts on this metric)


## The anchors and the thresholds

Measured once, GT-free, panel-independent. `floorRand` is the random-mask
floor, `neutralFloor` the content term (a fixed disk on every patch — what any
annotation inherits from the patches being similar), `ceilRef` the expert
pool's consensus reference. The fraction-below threshold is
mu_good - 1*sigma_good in each variant's own scale.

In [6]:
lines = ["| anchor | value |", "| --- | ---: |",
         f"| floorRand (32 random masks) | {record['anchors']['floorRand']:.4f} |",
         f"| neutralFloor (fixed disk) | {record['anchors']['neutralFloor']:.4f} |",
         f"| ceilRef (10-P1 consensus) | {record['anchors']['ceilRef']:.4f} |"]
print("\n".join(lines))
print()
print("fraction-below thresholds by variant:")
print(" | ".join(f"{k}: {v:.4f}" for k, v in record["thresholds"].items()))

| anchor | value |
| --- | ---: |
| floorRand (32 random masks) | 0.1167 |
| neutralFloor (fixed disk) | 0.1627 |
| ceilRef (10-P1 consensus) | 0.7995 |

fraction-below thresholds by variant:
N0: 0.7996 | S1: 1.0001 | S2z: -1.0000 | S2p: 0.6314 | S3: 1.0002


## The deliverable table — unit x normalization x summary over the P5 ladder

Decision metrics first, all zero-GT: dynamic range (score at c=0 minus c=10;
the compression is the problem, so width is the point), monotonicity
violations raw and beyond 2 combined SEs, the weakest adjacent step and the
9+1 vs 8+2 step (c=1 -> c=2) in SE units, and how many steps carry a verdict
past the declared floor. The Spearman column is VALIDATION ONLY — rank
correlation against the supervised mIoU of the consensus, reported, never a
selection input. `fracBelow` runs in the opposite direction (it GROWS with
contamination); its dynamic range is negative by construction and its
verdicts read inverted.

In [7]:
def arms_table(ladder_key):
    ladder = record["ladders"][ladder_key]
    rows = []
    u2 = ladder["u2Arm"]
    n_steps = len(u2["steps"])
    rows.append(("U2", "raw (prototype)", "-",
                 u2["dynamicRange"], u2["monotonicViolationsRaw"],
                 u2["monotonicViolationsBeyondNoise"], u2["minAdjacentZ"],
                 u2["step_1_2_z"], u2["spearmanVsSupervised"],
                 sum(1 for s in u2["steps"] if s["verdict"] == "separates"), n_steps))
    for variant in ("N0", "S1", "S2z", "S2p", "S3"):
        for summary in ("mean", "median", "q10", "fracBelow"):
            arm = ladder["arms"][variant][summary]
            rows.append(("U1", variant, summary, arm["dynamicRange"],
                         arm["monotonicViolationsRaw"], arm["monotonicViolationsBeyondNoise"],
                         arm["minAdjacentZ"], arm["step_1_2_z"], arm["spearmanVsSupervised"],
                         sum(1 for s in arm["steps"] if s["verdict"] == "separates"), n_steps))
    header = ("| unit | norm | summary | dynRange | viol | viol>2SE | minStepZ | "
              "step 1->2 z | Spearman (validation) | steps w/ verdict |",)
    lines = ["| unit | norm | summary | dynRange | viol | viol>2SE | minStepZ | "
             "step1-2z | rho (valid.) | verdicts |",
             "| --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"]
    for r in rows:
        rho = r[8]
        rho_s = "-" if rho is None else f"{rho:.3f}"
        lines.append(f"| {r[0]} | {r[1]} | {r[2]} | {r[3]:+.4f} | {r[4]} | {r[5]} | "
                     f"{r[6]:+.2f} | {r[7]:+.2f} | {rho_s} | {r[9]}/{r[10]} |")
    return "\n".join(lines)

print(arms_table("P5"))

| unit | norm | summary | dynRange | viol | viol>2SE | minStepZ | step1-2z | rho (valid.) | verdicts |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| U2 | raw (prototype) | - | +0.0074 | 0 | 0 | +1.28 | +2.44 | 1.000 | 7/10 |
| U1 | N0 | mean | +0.0291 | 0 | 0 | +8.97 | +11.94 | 1.000 | 8/10 |
| U1 | N0 | median | +0.0291 | 0 | 0 | +1.75 | +1.75 | 1.000 | 7/10 |
| U1 | N0 | q10 | +0.0346 | 0 | 0 | +1.07 | +26.83 | 1.000 | 3/10 |
| U1 | N0 | fracBelow | -0.8000 | 10 | 10 | -10.77 | -5.73 | -1.000 | 8/10 |
| U1 | S1 | mean | +0.0426 | 0 | 0 | +8.97 | +11.94 | 1.000 | 8/10 |
| U1 | S1 | median | +0.0427 | 0 | 0 | +1.75 | +1.75 | 1.000 | 7/10 |
| U1 | S1 | q10 | +0.0506 | 0 | 0 | +1.07 | +26.83 | 1.000 | 3/10 |
| U1 | S1 | fracBelow | -0.8000 | 10 | 10 | -10.77 | -5.73 | -1.000 | 8/10 |
| U1 | S2z | mean | +92.6237 | 0 | 0 | +8.97 | +11.94 | 1.000 | 8/10 |
| U1 | S2z | median | +92.6721 | 0 | 0 | +1.75 | +1.75 | 1.000 | 7/10 |
| U1 | S2z | q10 | +109.9899 | 0 | 0 | 

## The ladder itself — per-rung scores for the arms that matter

Mean +/- SE over the 30 draws per rung (the composition axis, the only
dispersion U1 has; the endpoints are single compositions and carry none).
The supervised mIoU column validates the ladder against quality. rungs 0 and
10 are deterministic; their SE is exactly 0.

In [8]:
def ladder_table(ladder_key):
    ladder = record["ladders"][ladder_key]
    sup = ladder["supervisedMeanByRung"]
    columns = [("U2 raw", [np.mean(ladder["u2ByDraw"][str(c)]) for c in range(11)],
                [np.std(ladder["u2ByDraw"][str(c)], ddof=1) / np.sqrt(len(ladder["u2ByDraw"][str(c)])) if len(ladder["u2ByDraw"][str(c)]) > 1 else 0.0 for c in range(11)])]
    for variant in ("N0", "S1", "S3"):
        for summary in ("mean", "q10"):
            arm = ladder["arms"][variant][summary]
            columns.append((f"U1 {variant}/{summary}", arm["means"], arm["se"]))
    lines = ["| c | " + " | ".join(f"{name} mean+-SE" for name, _, _ in columns) + " | sup. mIoU |",
             "| ---: | " + " | ".join("---:" for _ in columns) + " | ---: |"]
    for c in range(11):
        cells_ = []
        for _, means, ses in columns:
            cells_.append(f"{means[c]:.4f}+-{ses[c]:.4f}" if ses[c] > 0 else f"{means[c]:.4f} (det)")
        lines.append(f"| {c} | " + " | ".join(cells_) + f" | {sup[c]:.4f} |")
    return "\n".join(lines)

print(ladder_table("P5"))

| c | U2 raw mean+-SE | U1 N0/mean mean+-SE | U1 N0/q10 mean+-SE | U1 S1/mean mean+-SE | U1 S1/q10 mean+-SE | U1 S3/mean mean+-SE | U1 S3/q10 mean+-SE | sup. mIoU |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 0.7995 (det) | 0.7999 (det) | 0.7994 (det) | 1.0006 (det) | 0.9998 (det) | 1.0006 (det) | 0.9998 (det) | 0.9875 |
| 1 | 0.7993+-0.0001 | 0.7970+-0.0001 | 0.7964+-0.0001 | 0.9963+-0.0002 | 0.9954+-0.0002 | 0.9961+-0.0002 | 0.9951+-0.0002 | 0.9842 |
| 2 | 0.7991+-0.0001 | 0.7942+-0.0002 | 0.7747+-0.0008 | 0.9922+-0.0003 | 0.9637+-0.0012 | 0.9916+-0.0003 | 0.9611+-0.0012 | 0.9798 |
| 3 | 0.7989+-0.0001 | 0.7911+-0.0002 | 0.7706+-0.0007 | 0.9878+-0.0003 | 0.9577+-0.0010 | 0.9869+-0.0003 | 0.9546+-0.0010 | 0.9740 |
| 4 | 0.7985+-0.0001 | 0.7882+-0.0002 | 0.7688+-0.0005 | 0.9834+-0.0003 | 0.9551+-0.0007 | 0.9822+-0.0003 | 0.9519+-0.0008 | 0.9660 |
| 5 | 0.7980+-0.0001 | 0.7854+-0.0002 | 0.7677+-0.0005 | 0.9794+-0.0003 | 0.9534+-0.0007 | 0.9779+-0.0004 | 0.9501

## Early contamination — the first three rungs, quantile vs mean

Where the compression bit hardest is where a summary must earn its keep:
do the first contaminated panels already move? z = |delta| / combined SE
against rung 0, per summary, zero-GT. The lower quantile is the robustness
candidate; the mean is the conventional one.

In [9]:
def early_table(ladder_key):
    ladder = record["ladders"][ladder_key]
    u2 = ladder["u2Arm"]["earlyDetection"]
    lines = ["| arm | step 0->1 z | step 0->2 z | step 0->3 z |",
             "| --- | ---: | ---: | ---: |",
             f"| U2 raw | {u2['step_0_1']['z']:+.2f} | {u2['step_0_2']['z']:+.2f} | {u2['step_0_3']['z']:+.2f} |"]
    for variant in ("N0", "S1", "S2z", "S2p", "S3"):
        for summary in ("mean", "q10"):
            e = ladder["arms"][variant][summary]["earlyDetection"]
            lines.append(f"| U1 {variant}/{summary} | {e['step_0_1']['z']:+.2f} | "
                         f"{e['step_0_2']['z']:+.2f} | {e['step_0_3']['z']:+.2f} |")
    return "\n".join(lines)

print(early_table("P5"))

| arm | step 0->1 z | step 0->2 z | step 0->3 z |
| --- | ---: | ---: | ---: |
| U2 raw | +3.28 | +5.24 | +5.98 |
| U1 N0/mean | +20.21 | +30.51 | +42.36 |
| U1 N0/q10 | +21.05 | +31.08 | +43.27 |
| U1 S1/mean | +20.21 | +30.51 | +42.36 |
| U1 S1/q10 | +21.05 | +31.08 | +43.27 |
| U1 S2z/mean | +20.21 | +30.51 | +42.36 |
| U1 S2z/q10 | +21.05 | +31.08 | +43.27 |
| U1 S2p/mean | +10.36 | +18.35 | +25.65 |
| U1 S2p/q10 | +17.15 | +12.65 | +18.87 |
| U1 S3/mean | +20.21 | +30.51 | +42.36 |
| U1 S3/q10 | +21.05 | +31.08 | +43.27 |


## The P3 arm — systematic bias, the declared scope measurement

Reported apart, never hidden. A systematically biased annotator can be
consistent and not expert (assumption A6): on the P3 ladder the metric should
stay comparatively flat while the supervised quality degrades — that IS the
declared limit of its reach, not a defect of a variant. What is read here: the
score ladder's dynamic range and the supervised column's, side by side, plus
the same arms table for the P3 ladder.

In [10]:
p3 = record["ladders"]["P3"]
print("P3 ladder — the scope measurement")
print()
u2 = p3["u2Arm"]
print(f"U2 raw: dynamic range {u2['dynamicRange']:+.4f} over the ladder, "
      f"supervised mIoU moves {record['ladders']['P5']['supervisedMeanByRung'][0] - p3['supervisedMeanByRung'][-1]:+.4f} "
      "from c=0 to c=10")
print()
print(arms_table("P3"))

P3 ladder — the scope measurement

U2 raw: dynamic range +0.0014 over the ladder, supervised mIoU moves +0.1280 from c=0 to c=10

| unit | norm | summary | dynRange | viol | viol>2SE | minStepZ | step1-2z | rho (valid.) | verdicts |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| U2 | raw (prototype) | - | +0.0014 | 2 | 0 | -0.20 | +1.92 | 0.982 | 0/10 |
| U1 | N0 | mean | +0.0101 | 0 | 0 | +10.75 | +14.70 | 1.000 | 8/10 |
| U1 | N0 | median | +0.0101 | 0 | 0 | +1.75 | +1.75 | 1.000 | 7/10 |
| U1 | N0 | q10 | +0.0120 | 0 | 0 | +1.26 | +25.61 | 1.000 | 3/10 |
| U1 | N0 | fracBelow | -0.8000 | 10 | 10 | -10.77 | -5.73 | -1.000 | 8/10 |
| U1 | S1 | mean | +0.0148 | 0 | 0 | +10.75 | +14.70 | 1.000 | 8/10 |
| U1 | S1 | median | +0.0148 | 0 | 0 | +1.75 | +1.75 | 1.000 | 7/10 |
| U1 | S1 | q10 | +0.0175 | 0 | 0 | +1.26 | +25.61 | 1.000 | 3/10 |
| U1 | S1 | fracBelow | -0.8000 | 10 | 10 | -10.77 | -5.73 | -1.000 | 8/10 |
| U1 | S2z | mean | +32.0976 | 0 | 0 | +10.75 | +1

## Conclusions — generated from the record, provisional for r10

In [11]:
def conclusions(rec):
    """The closing read, derived from the record only. It must be able to come
    out different: everything below is computed, nothing is typed. Ranges are
    compared only WITHIN a scale: N0/S1/S3 are affine in raw C, S2z/S2p live in
    their own units — the scale-free read is the separation structure in SEs."""
    p5, p3 = rec["ladders"]["P5"], rec["ladders"]["P3"]
    out = []
    fid = rec["portFidelity"]
    ok = fid["maxAbsDiff"] is not None and fid["maxAbsDiff"] <= fid["tolerance"]
    out.append(f"1. Port fidelity: max |diff| vs the prototype's published table = "
               f"{fid['maxAbsDiff']:.2e} "
               f"({'within' if ok else 'ABOVE'} the declared tolerance of {fid['tolerance']:.0e}) — "
               + ("the port reproduces the prototype's compressed U2 ladder; every "
                  "comparison below is measured on the same material."
                  if ok else "the port runs on CPU and the prototype's device is unknown, "
                  "so the divergence is device/precision drift, reported per rung above; "
                  "the SHAPE reproduces and every comparison below is measured on the "
                  "same material, but the absolute levels carry that offset."))

    u2 = p5["u2Arm"]
    u1m = p5["arms"]["N0"]["mean"]
    out.append(f"2. The compression reproduces: U2 (the prototype's panel-consensus unit) "
               f"spans {u2['dynamicRange']:.4f} over the ladder "
               f"({u2['means'][0]:.4f} -> {u2['means'][-1]:.4f}), "
               f"{u2['monotonicViolationsBeyondNoise']} beyond-noise rebound(s), weakest "
               f"adjacent step {u2['minAdjacentZ']:+.2f} combined SEs.")

    out.append(f"3. The unit result (raw scale, same ladder, same draws): the per-annotator "
               f"unit U1/N0/mean spans {u1m['dynamicRange']:.4f} — "
               f"{u1m['dynamicRange'] / u2['dynamicRange']:.1f}x U2's span — with weakest "
               f"adjacent step {u1m['minAdjacentZ']:+.2f} SEs (U2: {u2['minAdjacentZ']:+.2f}) "
               f"and 9+1 vs 8+2 at {u1m['step_1_2_z']:+.2f} SEs (U2: {u2['step_1_2_z']:+.2f}). "
               f"{u1m['dynamicRange'] / u2['dynamicRange']:.1f}x and {u1m['minAdjacentZ'] / max(u2['minAdjacentZ'], 1e-9):.1f}x: "
               "the hypothesis that the compression is an artifact of measuring the "
               "consensus is what THIS run measured, not a selection.")

    z_identical = len({round(p5["arms"][v]["mean"]["minAdjacentZ"], 6)
                       for v in ("N0", "S1", "S3")}) == 1
    s1 = p5["arms"]["S1"]["mean"]
    s3 = p5["arms"]["S3"]["mean"]
    s2p = p5["arms"]["S2p"]["mean"]
    out.append(f"4. The normalization read: the affine family (N0/S1/S3) shares one "
               f"separation structure (min-step z identical: {z_identical}); S1/S3 only "
               f"re-express the same ladder as a fraction of the anchor span "
               f"({s1['dynamicRange']:.4f} and {s3['dynamicRange']:.4f} of it). "
               f"S2p's percentile compresses the tail (min step {s2p['minAdjacentZ']:+.2f} SEs, "
               f"{sum(1 for s_ in s2p['steps'] if s_['verdict'] == 'separates')}/10 verdicts) — "
               "a floor effect. One anchor finding the deliberation must see: S1/S3 land "
               "above 1.0 at c=0 because the consensus reference's own consistency "
               f"({rec['anchors']['ceilRef']:.4f}) sits BELOW the best annotators' "
               f"({max(rec['anchors']['popGood']):.4f}) — the neighborhood reference does "
               "not bound the scale's top.")

    q = p5["arms"]["N0"]["q10"]
    m = p5["arms"]["N0"]["mean"]
    fb = p5["arms"]["N0"]["fracBelow"]
    out.append(f"5. The summary read: the mean separates {sum(1 for s_ in m['steps'] if s_['verdict'] == 'separates')}/10 "
               f"steps across the whole ladder; the lower decile is the early alarm "
               f"(first step {q['earlyDetection']['step_0_1']['z']:+.1f} SEs) but SATURATES once the "
               f"decile fills with poor annotators ({sum(1 for s_ in q['steps'] if s_['verdict'] == 'separates')}/10 "
               f"verdicts, min step {q['minAdjacentZ']:+.2f} SEs); fracBelow runs inverted by design "
               f"(range {fb['dynamicRange']:+.2f}, direction opposite, declared per row).")

    sup_drop = p5["supervisedMeanByRung"][0] - p5["supervisedMeanByRung"][-1]
    p3m = p3["arms"]["N0"]["mean"]
    p3_pool = np.mean([rec["perAnnotatorScores"][f"P3|{a}"]
                       for a in range(rec["config"]["n_annotators"])])
    out.append(f"6. P3 (the scope limit, reported apart): on the bias ladder U2's range is "
               f"{p3['u2Arm']['dynamicRange']:+.4f} ({p3['u2Arm']['monotonicViolationsRaw']} raw "
               f"rebound(s), {sum(1 for s_ in p3['u2Arm']['steps'] if s_['verdict'] == 'separates')}/10 "
               f"verdicts) while the supervised mIoU drops {sup_drop:+.4f} across it — "
               "consistency does not track correctness. U1 reads the bias pool below the "
               f"experts (P1-P3 pool gap {np.mean(rec['anchors']['popGood']) - p3_pool:+.4f}) "
               f"and its bias ladder still opens ({p3m['dynamicRange']:+.4f}) — consistent bias "
               "ranks between the pools here, which is WHERE consistency puts it, not a "
               "claim that the metric detects wrongness.")

    out.append("7. Nothing is selected. The table stands per variant, provisional; the "
               "normalization and the panel summary close in the operator's r10 "
               "deliberation, and the margins above are what that deliberation reads.")
    return out

for line in conclusions(record):
    print(line)

1. Port fidelity: max |diff| vs the prototype's published table = 5.60e-03 (ABOVE the declared tolerance of 1e-03) — the port runs on CPU and the prototype's device is unknown, so the divergence is device/precision drift, reported per rung above; the SHAPE reproduces and every comparison below is measured on the same material, but the absolute levels carry that offset.
2. The compression reproduces: U2 (the prototype's panel-consensus unit) spans 0.0074 over the ladder (0.7995 -> 0.7921), 0 beyond-noise rebound(s), weakest adjacent step +1.28 combined SEs.
3. The unit result (raw scale, same ladder, same draws): the per-annotator unit U1/N0/mean spans 0.0291 — 4.0x U2's span — with weakest adjacent step +8.97 SEs (U2: +1.28) and 9+1 vs 8+2 at +11.94 SEs (U2: +2.44). 4.0x and 7.0x: the hypothesis that the compression is an artifact of measuring the consensus is what THIS run measured, not a selection.
4. The normalization read: the affine family (N0/S1/S3) shares one separation structur